In [3]:
import streamlit as st
from pathlib import Path

from langchain.agents import create_sql_agent
from langchain.sql_database import SQLDatabase

from langchain.agents.agent_types import AgentType
from langchain.callbacks import StreamlitCallbackHandler
from langchain.agents.agent_toolkits import SQLDatabaseToolkit
from sqlalchemy import create_engine
import sqlite3

from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
import os



# Title
st.set_page_config(page_title="The",page_icon="Icon")
st.title("CHAT-SQL")


local_db = "Use_local_db"
mysql = "Use_MySQL"
cloud_sql = "Use_Cloud_SQL"

## Radio button to select the database
radio_opt = ["Local Database", "MySQL", "Cloud SQL"]
db_option = st.sidebar.radio("Select the database", options=radio_opt)

if db_option == "MySQL":
    st.write("Please enter the details of the MySQL database")
    db_uri = mysql
    host = st.sidebar.text_input("Host", "localhost")
    user = st.sidebar.text_input("User", "root")
    password = st.sidebar.text_input("Password", "", type="password")
    mysql_db = st.sidebar.text_input("Database", "sample")

# If user wants to use sqlite database
elif db_option == "Local Database":
    db_uri = local_db
else:
    st.write("Please enter the details of the Cloud SQL database")


# API key for OpenAI and GROQ give option to user to select the API
api_option = ["OpenAI", "GROQ"]
api = st.sidebar.radio("Select the API", options=api_option)

if api == "OpenAI":
    st.write("Please enter the API key for OpenAI")
    api_key = st.sidebar.text_input("OpenAI API Key", "", type="password")
    os.environ['OPENAI_API_KEY']=api_key
    model = st.selectbox("Select the OpenAI model",["gpt-4o","gpt-4o-mini","o1","o1-mini"])
    openai = ChatOpenAI(model=model,streaming=True)
else:
    st.write("Please enter the API key for GROQ")
    api_key = st.sidebar.text_input("GROQ API Key", "", type="password")
    model = st.selectbox("Select the model",["llama3-8b-8192", "gemma2-9b-it", "mixtral-8x7b-32768","whisper-large-v3"])
    groq = ChatGroq(api_key=api_key, model=model,streaming=True)
    
if not db_uri:
    st.error("Please select the database")

if not api_key:
    st.error("Please enter the API key")

## Caching db info
@st.cache_resource(ttl="2h")
def configure_db(db_uri,host=None,user=None,password=None,mysql_db=None):
    if db_uri == "Local Database":
        dbfilepath = (Path(__file__).parent/"student.db").absolute()
        print(dbfilepath)
        createor = lambda:sqlite3.connect(f"file:{dbfilepath}?mode=ro",uri=True)
        return SQLDatabase(create_engine("sqllite:///",creator=createor))
    elif db_uri == "MySQL":
        if not (host and user and password and mysql_db):
            st.error("Please provide all MySQL connection details")
            st.stop()
        else:
            connection_string = f"mysql+mysqlconnector://{user}:{password}@{host}/{mysql_db}"
            sqlengine= create_engine(connection_string)
            return SQLDatabase(sqlengine)
    
if db_uri == "MySQL":
    db = configure_db(db_uri,host,user,password,mysql_db)
elif db_uri == "Local Database":
    db = configure_db(db_uri)


## Toolkit

toolkit = SQLDatabaseToolkit(db = db, llm = model)

agent = create_sql_agent(
    llm = model,
    toolkit = toolkit,
    verbose = True,
    agent_type = AgentType.zer, # gives response without previos context
    
)


## Creating session state to maintain chat history
if "message" not in st.session_state or st.sidebar.button("Clear chat"):
    st.session_state["message"] = [{"role":"assistant","context":"How can I help you?"}]
    
## Appending chat msgs
for msg in st.session_state.message:
    st.chat_message(msg["role"]).write(msg["content"])


## Asking the db
user_query  = st.chat_input(placeholder="Chat with your Database")


if(user_query):
    st.session_state.message.append({"role":"user","content":user_query})
    st.chat_message("user").write(user_query)
    
    with st.chat_message("assistant"):
        stream_callbacks = StreamlitCallbackHandler(st.container()) ## Display chain of though/working
        response = agent.run(user_query,callbacks=[stream_callbacks])
        st.session_state.message.append([{"role":"assistant","content":response}]) # Feeding the memory
        st.write(response)

SyntaxError: invalid syntax. Perhaps you forgot a comma? (315275180.py, line 122)